In [ ]:
!pip install -q transformers datasets

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
import torch
# from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Разделяем датасет так, чтобы в тестовой выборке было ровно 400 примеров

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')

# 1. Выделяем ровно 400 примеров в тестовую выборку, сохраняя стратификацию по метке
train_val_df, test_df = train_test_split(
    df,
    test_size=400,                # фиксированное количество примеров
    stratify=df['label'],
    random_state=42
)

print(f"Всего: {len(df)}, Тестовая: {len(test_df)}, Остаток (train+val): {len(train_val_df)}")

# 2. Делим оставшиеся (train_val_df) на обучающую и валидационную (10% от оставшихся идут в валидацию)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.10,               # 10% от train_val_df → validation
    stratify=train_val_df['label'],
    random_state=42
)

print(f"Обучающая: {len(train_df)}, Валидационная: {len(val_df)}")

# При желании сохраняем CSV-файлы
train_df.to_csv('/content/train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(  '/content/val.csv',   index=False, encoding='utf-8-sig')
test_df.to_csv( '/content/test.csv',  index=False, encoding='utf-8-sig')


In [ ]:
# df = pd.read_csv('/content/mydata.csv')  # должен быть загружен в Colab
# assert 'clean_text' in df.columns and 'label' in df.columns, "Требуются колонки clean_text и label"

In [ ]:
# train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
# val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
from torch.utils.data import Dataset
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch['clean_text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)
#Убираем ненужные текстовые поля и переименовываем
for split in ('train_tok', 'val_tok', 'test_tok'):
    ds = locals()[split]
    ds = ds.remove_columns(['raw_text', 'tokens', 'clean_text'])
    ds = ds.rename_column('label', 'labels')
    locals()[split] = ds

# Обёртка над HF Dataset → torch Dataset
class TorchDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        return {
            'input_ids':      torch.tensor(item['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'labels':         torch.tensor(item['labels'], dtype=torch.long),
        }

train_dataset = TorchDataset(train_tok)
val_dataset   = TorchDataset(val_tok)
test_dataset  = TorchDataset(test_tok)

# DataCollator для динамического паддинга
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/ruBERT_results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    logging_dir='/content/logs',
    logging_steps=50,
    report_to=["none"],

    seed=42
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
print(trainer.evaluate(test_tok))
preds = trainer.predict(test_tok)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)
print("\nClassification Report on Test Set:")
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from scipy.special import softmax
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# После получения preds и y_true, y_pred:
# preds = trainer.predict(test_tok)
# y_true = preds.label_ids
# y_pred = np.argmax(preds.predictions, axis=-1)

# 1. Матрица ошибок (confusion matrix)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['human', 'AI'])
disp.plot(cmap='Blues', values_format='d')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок (ruBERT)')
plt.tight_layout()
plt.show()

In [ ]:
# 2. ROC‐кривая
#    Применяем softmax к логитам, чтобы получить вероятности класса «1»
logits = preds.predictions  # shape = (n_samples, 2)
probs = softmax(logits, axis=1)[:, 1]  # вероятность класса 1

#    y_true уже содержит 0 и 1
fpr, tpr, thresholds = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)  # диагональная линия случайного классификатора
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая (ruBERT)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
history = pd.DataFrame(trainer.state.log_history)

train_loss = history[history['loss'].notna()][['epoch', 'loss']]

eval_metrics = history[history['eval_loss'].notna()][[
    'epoch', 'eval_loss', 'eval_accuracy', 'eval_precision', 'eval_recall', 'eval_f1'
]]

In [ ]:
plt.figure()
plt.plot(train_loss['epoch'], train_loss['loss'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss over Epochs')
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.plot(eval_metrics['epoch'], eval_metrics['eval_loss'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss over Epochs')
plt.grid(True)
plt.show()

In [ ]:
# Графики accuracy, precision, recall и F1
for metric in ['eval_accuracy', 'eval_precision', 'eval_recall', 'eval_f1']:
    plt.figure()
    plt.plot(eval_metrics['epoch'], eval_metrics[metric], marker='o')
    plt.xlabel('Epoch')
    plt.ylabel(metric.replace('eval_', '').capitalize())
    plt.title(f'{metric.replace("eval_", "").capitalize()} over Epochs')
    plt.grid(True)
    plt.show()

In [ ]:
from IPython.display import display
display(history)